# L3.2 · ACT: el encoder del VAE y la `z`


### El mapa de toda la serie

ACT es un **VAE condicional**:

- **Encoder del VAE** (sólo al entrenar): lee el estado y el bloque de acciones
  verdadero y lo resume en una variable `z`, el "estilo" de esa demostración.
- **Decoder = la policy** (al entrenar y al usar): un transformer
  encoder-decoder al estilo DETR. Lee la cámara, el estado y la `z`, y escribe
  el bloque entero de acciones de una pasada.
- **Al usarlo:** el encoder del VAE se tira y `z = 0`.

| Notebook | Qué abre |
|---|---|
| L3.1 | Los datos: de la grabación al ejemplo de entrenamiento |
| L3.2 | El encoder del VAE y la `z` |
| L3.3 | El decoder: la policy por dentro |
| L3.4 | El entrenamiento: un paso desarmado, y el modelo aprendiendo |
| L3.5 | El modelo en uso: ejecutar los bloques |

Todo usa el código real de LeRobot 0.4.1 y **nuestro modelo entrenado**
(`act_cheap_v1`), desarmado con `toolkit/policy/act_internals.py`. Sus tests
verifican que las piezas encadenadas dan exactamente lo mismo que el modelo.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy import act_internals as act
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
policy, model, config = loaded.policy, loaded.policy.model, loaded.policy.config
_ = policy.eval()

## 1. Para qué existe

Si se entrena una regresión directa (observación → acciones), ante
demostraciones con distintos estilos la red aprende **el promedio**: un
movimiento borroso que no hizo nadie.

La idea del VAE: durante el entrenamiento, alguien que **ve la respuesta**
(el encoder) le pasa al decoder un resumen de "cómo lo hizo el humano esta
vez", la `z`. Con esa pista, el decoder no tiene que promediar estilos.

- **Autoencoder**: lo que entra al encoder (el bloque de acciones verdadero)
  es lo mismo que el decoder tiene que reconstruir.
- **Condicional**: el decoder además ve la observación.
- **Variacional**: el encoder no devuelve un punto sino una campana (centro y
  ancho), y la `z` se sortea de ahí. El término **KL** empuja todas las
  campanas hacia la campana estándar, para que al usar el modelo se pueda
  poner `z = 0` y caer en un estilo válido.

El paper lo considera esencial con datos humanos: sin el VAE, el éxito en sus
tareas simuladas con demostraciones humanas cayó de 35,3% a 2% (con datos de
un guión determinista no hubo diferencia).

## 2. Qué le entra: un lote real

In [ ]:
from lerobot.datasets.factory import resolve_delta_timestamps
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

meta = LeRobotDatasetMetadata("local/pick_mameluco_act_v1", root=ROOT)
dataset = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav",
                         delta_timestamps=resolve_delta_timestamps(config, meta))
loader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True, generator=torch.Generator().manual_seed(0))
batch = loaded.preprocessor(next(iter(loader)))
print("estado", tuple(batch["observation.state"].shape), "· bloque", tuple(batch["action"].shape))

## 3. Los tokens: `[CLS, estado, acción_1 … acción_48]`

Son **50 tokens** en nuestro modelo (102 en el paper, con bloques de 100). La
imagen **no** entra: los autores la sacaron para entrenar más rápido.

- **CLS**: un token aprendido, igual en todos los ejemplos, que empieza
  vacío. Su trabajo es juntar el resumen.
- **Estado**: las 26 articulaciones × una matriz → 1 token de 192.
- **Acciones**: cada una de las 48 acciones × **otra** matriz, compartida entre
  las 48 → 48 tokens de 192. La matriz del estado y la de las acciones son
  distintas.

A cada token se le suma una **posición fija** (senoidal, no aprendida), para
que el transformer sepa cuál es el CLS, cuál el estado y en qué orden van las
acciones.

In [ ]:
print("CLS                 :", tuple(model.vae_encoder_cls_embed.weight.shape))
print("matriz del estado   :", tuple(model.vae_encoder_robot_state_input_proj.weight.T.shape), "(26 → 192)")
print("matriz de acciones  :", tuple(model.vae_encoder_action_input_proj.weight.T.shape), "(26 → 192, la misma para las 48)")
positions = model.vae_encoder_pos_enc[0].numpy()
print("posiciones fijas    :", positions.shape)

fig, ax = plt.subplots(figsize=(12, 3))
ax.imshow(positions.T, aspect="auto", cmap="RdBu")
ax.set_xlabel("token (0 = CLS, 1 = estado, 2..49 = acciones)"); ax.set_ylabel("dimensión")
ax.set_title("posiciones senoidales: cada token tiene su 'código de barras'")
plt.tight_layout()

Los tokens de relleno (`action_is_pad`) se **enmascaran**: nadie los puede
mirar en la atención.

## 4. Adentro: 4 capas de atención completa

Atención completa (no causal): cada token mira a todos. Cada salida es el
mismo token enriquecido con lo que vio de los demás. De las 50 salidas, **sólo
se usa la del CLS**; las otras 49 se tiran.

¿Cómo aprende el CLS a resumir bien? Todo se entrena junto: el error del
decoder viaja hacia atrás por la `z` hasta el CLS. Si el CLS no junta algo
útil, el decoder reconstruye peor.

Miramos a quién atiende el CLS en cada capa:

In [ ]:
with act.capture_attention(model) as weights:
    style = act.encode_style(model, batch)
fig, axes = plt.subplots(1, config.n_vae_encoder_layers, figsize=(16, 2.8), sharey=True)
for layer, ax in enumerate(axes):
    cls_row = weights[f"vae_encoder.layers.{layer}.self_attn"][:, 0, :].mean(0).numpy()
    ax.bar(range(50), cls_row, color=["k"] + ["tab:orange"] + ["tab:blue"] * 48)
    ax.set_title(f"capa {layer}"); ax.set_xlabel("token")
axes[0].set_ylabel("atención del CLS")
plt.suptitle("a quién mira el CLS (negro: CLS, naranja: estado, azul: acciones 1..48)", y=1.05); plt.tight_layout()

En un encoder que funciona, el CLS repartiría su atención entre las acciones
para resumirlas. En el nuestro, en las primeras capas se mira casi sólo **a sí
mismo**, y en la última reparte parejo entre todos: no está buscando nada en
particular. Es otra cara del colapso que vemos en la sección 7.

## 5. Del CLS a la `z`

1. La salida del CLS (192 números) pasa por una capa lineal → **64**.
2. Se parte al medio: los primeros 32 son los **centros** (μ) y los segundos
   32 el **log de la varianza** (LeRobot lo llama `log_sigma_x2` = 2·log σ).
   Se devuelve el logaritmo porque así la red puede largar cualquier número,
   positivo o negativo, y la varianza igual sale positiva.
3. **Sorteo con el truco de reparametrización**: z = μ + σ · ε, con ε sorteado
   de una normal estándar. El azar queda "afuera" (en ε), así el error puede
   viajar hacia atrás por μ y σ y entrenarlos.

In [ ]:
print("μ :", tuple(style.mu.shape), "· log σ² :", tuple(style.log_sigma_x2.shape))
z = act.sample_latent(style)
print("z :", tuple(z.shape), "→ una z de 32 números por ejemplo")

## 6. El KL

Para cada dimensión, el KL mide cuánto se aleja la campana (μ, σ) de la
estándar (0, 1):

```latex
\mathrm{KL} = -\tfrac{1}{2} \sum_{d=1}^{32} \left(1 + \log\sigma_d^2 - \mu_d^2 - \sigma_d^2\right)
```

Vale 0 sólo si μ = 0 y σ = 1: o sea, si la campana es la estándar y **no dice
nada del ejemplo**. Ese es el tironeo del VAE: la reconstrucción quiere que la
`z` cuente cosas, y el KL (multiplicado por `kl_weight` = 10) cobra por cada
cosa que cuenta.

## 7. Lo que hace nuestro modelo: colapsó

Pasamos 64 bloques reales, de distintos episodios y momentos. Si el encoder
funcionara, cada bloque tendría su propio μ.

In [ ]:
mu, sigma, kl = style.mu.numpy(), style.sigma.numpy(), style.kl_per_sample().numpy()
fig, axes = plt.subplots(1, 3, figsize=(15, 3.2))
axes[0].scatter(mu[:, 0], mu[:, 1], s=12); axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].set_title("μ de 64 bloques (dimensiones 0 y 1)"); axes[0].set_aspect("equal")
axes[1].hist(sigma.ravel(), bins=40); axes[1].set_title("σ de todas las dimensiones")
axes[2].hist(kl, bins=30); axes[2].set_title("KL por ejemplo")
plt.tight_layout()
print(f"|μ| medio {np.abs(mu).mean():.5f} · σ medio {sigma.mean():.4f} · KL medio {kl.mean():.1e}")

Todos los bloques caen en el mismo punto: μ ≈ 0 y σ ≈ 1. El encoder devuelve
**siempre la campana estándar**, sin importar qué hizo el operador. La `z` que
recibe el decoder es ruido puro, y por eso el decoder aprendió a ignorarla.
Esto se llama ***posterior collapse***.

¿Por qué pasa? Si el decoder puede predecir bien sólo con la observación (un
solo operador, muy consistente: la velocidad varía 9% entre episodios,
notebook 04), la `z` aporta poco. Y usarla tiene un precio, el KL × 10.
Entonces lo más barato para el optimizador es apagarla. En L3.4 se ve
**cuándo** pasa durante el entrenamiento.

La comprobación del otro lado: si el decoder usara la `z`, cambiarla debería
cambiar el bloque.

In [ ]:
plain = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
frame = plain[3000]
obs = loaded.preprocessor({k: frame[k] for k in ["observation.state", CAMERA]})

def chunk_with_latent(latent):
    scene = act.scene_tokens(model, obs, latent)
    _, actions = act.decode_chunk(model, act.encode_scene(model, scene), scene)
    return loaded.postprocessor(actions)[0]

base = chunk_with_latent(act.zero_latent(model, 1, "cpu"))
for scale in [1, 3, 10]:
    other = chunk_with_latent(torch.randn(1, config.latent_dim) * scale)
    print(f"z con desvío {scale:2d}: el bloque cambia {(other - base).abs().mean():.5f} rad en promedio")

Con `z` del tamaño para el que se entrenó (desvío 1), el bloque cambia una
diezmilésima de radián. **El decoder ignora la `z`.**

---

## Para profundizar

- **Apagar el VAE es una opción de LeRobot**: `--policy.use_vae=false`. La
  policy queda como regresión pura con bloques: el mismo transformer, sin el
  token de la `z`, entrenado sólo con L1. Con nuestro modelo, en la práctica,
  es lo que ya está pasando.
- **Bajar `kl_weight`** cobra menos por usar la `z`. Es el experimento natural
  para ver si deja de colapsar (L3.4 dice cómo).
- El colapso no es raro en VAEs con decoders potentes: si el decoder puede
  hacer el trabajo solo, no necesita la pista.